# Dispatch Arena - GRPO Training on Colab

**OpenEnv / DispatchArena training template**

Train a delivery-dispatch agent for `DispatchArena` using **GRPO** with Hugging Face TRL. This notebook mirrors the structure of the `kube_sre_gym` Colab notebook, but adapts it to a local, deterministic dispatch environment with `mini` and `normal` modes.

| Component | Detail |
|---|---|
| Environment | `DispatchArenaEnvironment` from this repo |
| Training | This Colab notebook |
| Model | `Qwen/Qwen3-0.6B` or `Qwen/Qwen3-1.7B` + LoRA |
| Framework | HF TRL GRPO |
| Modes | `mini` for curriculum, `normal` for dispatcher training |

> Note: `DispatchArena` does not yet ship a dedicated `dispatch_arena.train` module, so this notebook defines the tool wrapper, prompt, and reward helpers inline.

## 1. Install Dependencies

Clone the repo, install the `dispatch_arena` package from the `dispatch_arena/` subdirectory, and install the RL / plotting stack.

This notebook is preconfigured for the public GitHub repo and the deployed HF Space:

- GitHub: `https://github.com/FReakYdiVi/DispatchArena`
- HF Space: `https://freakdivi-dispatch-arena-v0.hf.space`

You can still override the repo URL manually if you fork the project later.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/FReakYdiVi/DispatchArena.git"
BRANCH = "main"
WORKDIR = "/content/DispatchArena"
INSTALL_VLLM = False  # Set True on A100 / stronger Colab runtimes if desired.
HF_SPACE_URL = "https://freakdivi-dispatch-arena-v0.hf.space"

if not os.path.exists(WORKDIR):
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        BRANCH,
        REPO_URL,
        WORKDIR,
    ])

base_deps = [
    "peft>=0.15.0",
    "transformers>=4.51.0",
    "datasets>=3.0.0",
    "accelerate>=1.0.0",
    "huggingface_hub>=0.20.0",
    "matplotlib",
    "numpy<2",
    "pandas",
    "packaging",
]
trl_pkg = "trl[vllm]>=0.19.0" if INSTALL_VLLM else "trl>=0.19.0"
deps = [trl_pkg, *base_deps]
if INSTALL_VLLM:
    deps.append("vllm>=0.11.0")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *deps])

from packaging import version
try:
    import torch
    torch_version = version.parse(torch.__version__.split("+")[0])
except Exception:
    torch = None
    torch_version = version.parse("0")

if torch is None or torch_version < version.parse("2.4.0"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch>=2.4.0"])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    os.path.join(WORKDIR, "dispatch_arena"),
])

os.chdir(WORKDIR)
print(f"Repo ready at: {WORKDIR}")
print(f"HF Space : {HF_SPACE_URL}")

## 2. Configuration

Set the model, environment difficulty, and training hyperparameters. This notebook trains directly against the Python environment class, so you do **not** need a remote HF Space just to smoke test or train locally in Colab.

In [ ]:
import os
from datetime import datetime
from pathlib import Path

if "WORKDIR" not in globals():
    WORKDIR = "/content/DispatchArena"
if "HF_SPACE_URL" not in globals():
    HF_SPACE_URL = "https://freakdivi-dispatch-arena-v0.hf.space"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

MODEL_ID = "Qwen/Qwen3-0.6B"
HUB_REPO = "Freakdivi/dispatch-arena-qwen3-0.6b"  # optional adapter repo if you want to push LoRA later
HF_SPACE_URL = "https://freakdivi-dispatch-arena-v0.hf.space"

TRAIN_MODE = "normal"
TRAIN_BUCKET = "easy"
VISIBLE_PREP = False
NUM_COURIERS = 3
NUM_ORDERS = 5
MAX_TICKS = 18

MAX_STEPS = 16
NUM_GENERATIONS = 2
MAX_COMPLETION_LENGTH = 256
MAX_TOOL_CALLING_ITERATIONS = 12
LEARNING_RATE = 1e-5
SEED = 7
USE_VLLM = False

OUTPUT_DIR = Path(WORKDIR) / "dispatch_arena" / "outputs" / f"dispatch-arena-grpo-{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model       : {MODEL_ID}")
print(f"Mode        : {TRAIN_MODE}")
print(f"Bucket      : {TRAIN_BUCKET}")
print(f"Max steps   : {MAX_STEPS}")
print(f"Generations : {NUM_GENERATIONS}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"HF Space    : {HF_SPACE_URL}")

## 3. Smoke Test the Environment

Run one deterministic normal-mode episode reset and a single heuristic action. This verifies that the package import, scenario generation, and transition function all work before any training starts.

In [ ]:
from dispatch_arena import Config, DispatchArenaEnvironment, Mode


def heuristic_action(observation):
    state = observation.state
    if state.mode.value == "normal":
        courier = next((c for c in state.couriers if c.status.value == "idle" and c.load is None), None)
        order = next((o for o in state.orders if o.status.value in {"queued", "ready"} and o.assigned_courier_id is None), None)
        if courier and order:
            return {"action_type": "assign", "courier_id": courier.id, "order_id": order.id}
        return {"action_type": "hold"}

    for action in ["pickup", "dropoff", "go_pickup", "go_dropoff", "wait"]:
        if action in observation.legal_actions:
            return action
    return observation.legal_actions[0]


config = Config(
    mode=Mode(TRAIN_MODE),
    scenario_bucket=TRAIN_BUCKET,
    visible_prep=VISIBLE_PREP,
    num_couriers=NUM_COURIERS,
    num_orders=NUM_ORDERS,
    max_ticks=MAX_TICKS,
)

env = DispatchArenaEnvironment(config=config)
obs = env.reset(seed=SEED)
print("Initial observation:")
print(obs.summary_text)
print("Legal actions:", obs.legal_actions)
print("Reward breakdown:", obs.reward_breakdown.to_dict())

first_action = heuristic_action(obs)
print("\nFirst heuristic action:", first_action)
obs = env.step(first_action)
print("After one step:")
print(obs.summary_text)
print("Events:", obs.info.get("events", []))
print("Reward breakdown:", obs.reward_breakdown.to_dict())

## 4. Define the Tool Wrapper and Reward Helpers

`DispatchArena` already has typed actions and decomposed rewards. For GRPO, we expose a thin tool wrapper with semantic methods like `assign`, `reposition`, `hold`, and `prioritize`. The reward functions then read the latest `reward_breakdown` from the environment wrapper.

In [ ]:
from dispatch_arena import Action, Config, DispatchArenaEnvironment, Mode

SYSTEM_PROMPT = """You are an AI dispatch operator for food and grocery delivery.
Your goal is to maximize completed deliveries and on-time bonus while avoiding invalid actions,
idle time, route churn, and unfair courier allocation.

Rules:
- Use the available tools instead of free-form reasoning.
- Prefer valid actions that move the shift forward.
- If no useful assignment or reposition is available, use hold.
- Pay attention to legal actions, courier status, order status, and deadlines.
- Keep decisions concise and tool-oriented.
"""


def format_observation(observation):
    state = observation.state
    courier_lines = [
        f"- {c.id}: node={c.node_id}, status={c.status.value}, eta={c.eta_remaining}, assigned={c.assigned_order_id}, load={c.load}"
        for c in state.couriers
    ]
    order_lines = [
        f"- {o.id}: kind={o.kind}, pickup={o.pickup_node_id}, dropoff={o.dropoff_node_id}, status={o.status.value}, deadline={o.deadline_tick}, assigned={o.assigned_courier_id}"
        for o in state.orders
    ]
    return "\n".join([
        f"Summary: {observation.summary_text}",
        f"Done: {observation.done}",
        f"Verifier: {observation.verifier_status.value}",
        f"Legal actions: {', '.join(observation.legal_actions)}",
        "Couriers:",
        *courier_lines,
        "Orders:",
        *order_lines,
        f"Reward breakdown: {observation.reward_breakdown.to_dict()}",
    ])


class DispatchToolEnv:
    """Thin TRL/OpenEnv-style wrapper around the local DispatchArena environment."""

    def __init__(self):
        self.env = None
        self.last_observation = None
        self.last_config = None

    def reset(
        self,
        seed=0,
        mode="normal",
        scenario_bucket="easy",
        visible_prep=False,
        num_couriers=3,
        num_orders=5,
        max_ticks=18,
        **kwargs,
    ):
        """Reset the dispatch shift and return the initial public observation."""
        self.last_config = Config(
            mode=Mode(mode),
            scenario_bucket=scenario_bucket,
            visible_prep=bool(visible_prep),
            num_couriers=int(num_couriers),
            num_orders=int(num_orders),
            max_ticks=int(max_ticks),
        )
        self.env = DispatchArenaEnvironment(config=self.last_config)
        self.last_observation = self.env.reset(seed=int(seed), config=self.last_config)
        return format_observation(self.last_observation)

    def view_state(self):
        """Return the latest visible dispatch state."""
        return format_observation(self.last_observation)

    def assign(self, courier_id: str, order_id: str):
        """Assign an idle courier to an unassigned order."""
        self.last_observation = self.env.step(Action(action_type="assign", courier_id=courier_id, order_id=order_id))
        return format_observation(self.last_observation)

    def reposition(self, courier_id: str, node_id: str):
        """Move an idle courier to a different graph node."""
        self.last_observation = self.env.step(Action(action_type="reposition", courier_id=courier_id, node_id=node_id))
        return format_observation(self.last_observation)

    def hold(self, courier_id: str | None = None):
        """Advance one tick without assigning a new order."""
        payload = {"action_type": "hold"}
        if courier_id:
            payload["courier_id"] = courier_id
        self.last_observation = self.env.step(payload)
        return format_observation(self.last_observation)

    def prioritize(self, order_id: str | None = None):
        """Record dispatcher intent for an open order without changing hidden state."""
        payload = {"action_type": "prioritize"}
        if order_id:
            payload["order_id"] = order_id
        self.last_observation = self.env.step(payload)
        return format_observation(self.last_observation)

    @property
    def metrics(self):
        if self.last_observation is None:
            return {}
        breakdown = self.last_observation.reward_breakdown
        summary = self.env.get_episode_summary() if self.env is not None else {}
        return {
            "reward": self.last_observation.reward,
            "total_reward": self.last_observation.state.total_reward,
            "service_reward": breakdown.success_reward + breakdown.on_time_bonus + breakdown.late_penalty,
            "validity_reward": breakdown.invalid_penalty,
            "efficiency_reward": breakdown.step_cost + breakdown.idle_penalty + breakdown.route_churn_penalty,
            "fairness_reward": breakdown.fairness_penalty,
            "done": self.last_observation.done,
            **summary,
        }


def reward_total(environments, **kwargs):
    return [env.metrics.get("total_reward", 0.0) for env in environments]

## 5. Configure GRPO Training

Unlike `kube_sre_gym`, this notebook does not rely on a package-provided `rollout_once()` helper. Instead, it uses TRL's tool-based environment integration through `environment_factory=DispatchToolEnv`.

Each training example is just a short dispatch instruction; the environment carries the actual simulator state across tool calls.

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime not detected. In Colab, switch to Runtime > Change runtime type > GPU before training.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_prompts = [
    "You are the dispatcher for a live food and grocery delivery shift. Use the available tools to maximize deliveries and on-time bonus while minimizing invalid actions, idle time, route churn, and fairness imbalance."
] * max(MAX_STEPS, 8)
train_dataset = Dataset.from_dict({"prompt": train_prompts})

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_tool_calling_iterations=MAX_TOOL_CALLING_ITERATIONS,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    beta=0.0,
    log_completions=True,
    use_vllm=USE_VLLM,
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total],
    train_dataset=train_dataset,
    args=grpo_config,
    peft_config=lora_config,
    environment_factory=DispatchToolEnv,
)

print("GRPOTrainer initialized.")
print(f"Dataset size: {len(train_dataset)}")

## 6. Train

Launch GRPO training. The model will interact with `DispatchToolEnv`, which wraps the real `DispatchArenaEnvironment` and exposes semantic tools like `assign`, `reposition`, `hold`, and `prioritize`.

In [ ]:
print("Starting GRPO training...")
print(f"Model       : {MODEL_ID}")
print(f"Mode        : {TRAIN_MODE}")
print(f"Bucket      : {TRAIN_BUCKET}")
print(f"Max steps   : {MAX_STEPS}")
print(f"Generations : {NUM_GENERATIONS}")
print()

train_output = trainer.train()
FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_lora"
trainer.save_model(str(FINAL_ADAPTER_DIR))

print("\nTraining metrics:", train_output.metrics)
print(f"Saved final adapter to: {FINAL_ADAPTER_DIR}")

## 7. Inspect Training Logs

Plot any reward-related values that appear in `trainer.state.log_history`. Depending on the TRL version, the exact metric names may differ slightly, so this cell auto-detects available reward columns.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
display(logs.tail())
logs.to_csv(OUTPUT_DIR / "train_log_history.csv", index=False)

reward_cols = [
    col for col in logs.columns
    if isinstance(col, str) and ("reward" in col.lower() or col.lower().startswith("eval_"))
]
loss_cols = [
    col for col in logs.columns
    if isinstance(col, str) and "loss" in col.lower()
]

if reward_cols:
    ax = logs[reward_cols].plot(figsize=(12, 6), title="DispatchArena GRPO reward curves")
    ax.set_xlabel("log step")
    ax.set_ylabel("value")
    reward_plot_path = PLOTS_DIR / "reward_curve.png"
    plt.savefig(reward_plot_path, bbox_inches="tight")
    plt.show()
    print(f"Saved reward plot to: {reward_plot_path}")
else:
    print("No reward columns found in trainer.state.log_history. Inspect logs manually:")
    display(logs.tail(20))

if loss_cols:
    ax = logs[loss_cols].plot(figsize=(12, 6), title="DispatchArena GRPO loss curves")
    ax.set_xlabel("log step")
    ax.set_ylabel("value")
    loss_plot_path = PLOTS_DIR / "loss_curve.png"
    plt.savefig(loss_plot_path, bbox_inches="tight")
    plt.show()
    print(f"Saved loss plot to: {loss_plot_path}")
else:
    print("No loss columns found in trainer.state.log_history.")

## 8. Optional: Heuristic Baseline Rollout

Use this cell to compare the built-in heuristic against future trained checkpoints on the same seeds. It is a simple sanity baseline, not a learned agent.

In [ ]:
def run_heuristic_episode(seed=7, mode="normal", bucket="easy"):
    config = Config(
        mode=Mode(mode),
        scenario_bucket=bucket,
        visible_prep=False,
        num_couriers=NUM_COURIERS,
        num_orders=NUM_ORDERS,
        max_ticks=MAX_TICKS,
    )
    env = DispatchArenaEnvironment(config=config)
    obs = env.reset(seed=seed)
    while not obs.done:
        obs = env.step(heuristic_action(obs))
    return env.get_episode_summary()

baseline = run_heuristic_episode(seed=SEED, mode=TRAIN_MODE, bucket=TRAIN_BUCKET)
baseline

## 9. Notes / Next Steps

- GitHub repo used by this notebook: `https://github.com/FReakYdiVi/DispatchArena`
- Deployed HF Space: `https://freakdivi-dispatch-arena-v0.hf.space`
- This notebook trains against the local Python environment class for speed, but the same prompts and tools can be adapted to the deployed Space later through `DispatchArenaClient(base_url=HF_SPACE_URL)`.
- For hackathon submission evidence, upload `train_log_history.csv`, `plots/reward_curve.png`, and `plots/loss_curve.png` from `OUTPUT_DIR` into `dispatch_arena/outputs/` in the repo after a real Colab run.
- Once the notebook is stable, you can factor the inline helpers into a real `dispatch_arena.train` module, exactly like the `kube_sre_gym` example.